# DARKEN Fusion Reanalysis with CalciMetrics 2.0

Reanalysis of 2 iPSC fusion recordings (experiments 870 and 977) from the DARKEN sDARKEN characterisation dataset.  
This notebook reproduces the original analysis from `all_fusion/imaging_analysis_darken_fusion_all.ipynb` using the CalciMetrics 2.0 API.

**Pipeline** (identical to the original):  
`raw CSV` → background subtraction → ΔF/F (median baseline) → exclude bad ROIs → combine all ROIs → linear detrend → Gaussian smooth (window=5, σ=2.5) → % ΔF/F

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from calcimetrics import Dataset, load_imagej_csv
from calcimetrics.core import Recording
from calcimetrics.normalization import compute_dff, to_percent
from calcimetrics.detrending import detrend_linear
from calcimetrics.smoothing import gaussian_smooth
from calcimetrics.plotting import plot_overlay, plot_mean_with_error, plot_heatmap


In [ ]:
# ── Plot style ───────────────────────────────────────────────────────────
# Set USE_NATURE_STYLE = True for a publication-ready Nature-journal look.
# Requires SciencePlots:  pip install SciencePlots
USE_NATURE_STYLE = True

import matplotlib as mpl

def apply_style():
    if USE_NATURE_STYLE:
        try:
            import scienceplots  # noqa: F401
            plt.style.use(["science", "nature", "no-latex"])
            mpl.rcParams.update({
                "figure.dpi":      150,
                "font.size":       9,
                "axes.labelsize":  10,
                "axes.titlesize":  10,
                "xtick.labelsize": 8,
                "ytick.labelsize": 8,
                "legend.fontsize": 8,
                "lines.linewidth": 1.0,
            })
        except ImportError:
            print("SciencePlots not found — install with:  pip install SciencePlots")
    else:
        plt.style.use("default")

apply_style()
print(f"Style: {'science + nature + no-latex' if USE_NATURE_STYLE else 'default'}")

# ── Figure export ─────────────────────────────────────────────────────────
SAVE_FIGURES = True
FIGURES_DIR  = Path.cwd().parent / "figures" / "darken_fusion"

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(exist_ok=True)
    print(f"Figures will be saved to: {FIGURES_DIR}")

def save_fig(fig, name: str) -> None:
    if not SAVE_FIGURES:
        return
    for ext in ("png", "svg"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"  saved  {name}.png / .svg")


## 1. Load recordings

Exclude lists verified against `all_fusion/cleaned/` CSVs:

| Experiment | Raw ROIs | Excluded | Kept |
|---|---|---|---|
| 870 | 11 | — (none) | 11 |
| 977 | 17 | Mean1, Mean3, Mean7, Mean8, Mean9, Mean10 | 11 |

In [ ]:
ROOT      = Path.cwd().parent
data_root = ROOT / "input_data" / "fusion"

configs = {
    "870": {
        "file":    "Results_870.csv",
        "exclude": [],  # all 11 ROIs kept
        "bio_rep": "fusion_A",
    },
    "977": {
        "file":    "Results_977.csv",
        "exclude": ["Mean1", "Mean3", "Mean7", "Mean8", "Mean9", "Mean10"],
        "bio_rep": "fusion_B",
    },
}


In [ ]:
fusion = Dataset()

for rec_id, cfg in configs.items():
    path = data_root / rec_id / cfg["file"]
    rec = load_imagej_csv(
        path, fps=0.5, name=rec_id,
        biological_replicate=cfg["bio_rep"],
        condition="sDARKEN_fusion",
    )
    rec.traces = compute_dff(rec.traces, method="median")
    if cfg["exclude"]:
        rec = rec.exclude_rois(cfg["exclude"])
    fusion.add(rec)
    print(f"  {rec_id}: {rec.traces.shape[1]} ROIs — {list(rec.traces.columns)}")

print()
print(fusion.summary())


## 2. Combine, detrend, smooth

All ROIs from both recordings are pooled into one DataFrame (`level="roi"` = 22 individual cells).  
Pipeline: linear detrend → Gaussian smooth (window = 5 frames, σ = 2.5) → % ΔF/F

In [ ]:
# Pool all 22 ROIs (601 frames × 22 columns).
all_rois = fusion.combine_traces(level="roi")

fps = 0.5
all_rois = detrend_linear(all_rois)
all_rois = gaussian_smooth(all_rois, window_size=5, sigma=2.5)
all_rois = to_percent(all_rois)

n_rois = all_rois.shape[1]

all_rec = Recording(
    name=f"all fusion (n = {n_rois} ROIs)",
    raw=all_rois,
    traces=all_rois,
    fps=fps,
    excluded_rois=[],
    biological_replicate="pooled",
    condition="sDARKEN_fusion",
)

print(f"Shape: {all_rois.shape}  "
      f"Time: 0–{all_rec.duration_s:.0f} s  "
      f"Value range: {all_rois.values.min():.1f} – {all_rois.values.max():.1f} % ΔF/F")


## 3. Plots — all fusion ROIs pooled

Three figures that reproduce the outputs saved in `all_fusion/cleaned/`,  
using `calcimetrics.plotting`.

In [ ]:
fig = plot_overlay(all_rec, ylabel="% ΔF/F", alpha=0.5)
save_fig(fig, "01_all_rois_overlay")
plt.show()


In [ ]:
fig = plot_mean_with_error(all_rec, error="sd", ylabel="% ΔF/F")
save_fig(fig, "02_all_rois_mean_sd")
plt.show()


In [ ]:
fig = plot_heatmap(all_rec, cbar_label="% ΔF/F", figsize=(10, 8))
save_fig(fig, "03_all_rois_heatmap")
plt.show()


## 4. Per-recording plots

Spontaneous serotonin release events are not synchronised across recordings, so per-recording plots show the individual dynamics more clearly.

In [ ]:
for rec_id, rec in fusion.recordings.items():
    traces = detrend_linear(rec.traces.copy())
    traces = gaussian_smooth(traces, window_size=5, sigma=2.5)
    traces = to_percent(traces)

    proc_rec = Recording(
        name=f"Fusion {rec_id}  ({traces.shape[1]} ROIs)",
        raw=traces,
        traces=traces,
        fps=rec.fps,
        excluded_rois=rec.excluded_rois,
        biological_replicate=rec.biological_replicate,
        condition=rec.condition,
    )

    fig = plot_overlay(proc_rec, ylabel="% ΔF/F", alpha=0.6)
    save_fig(fig, f"04_fusion_{rec_id}_overlay")
    plt.show()

    fig = plot_mean_with_error(proc_rec, error="sd", ylabel="% ΔF/F")
    save_fig(fig, f"05_fusion_{rec_id}_mean_sd")
    plt.show()


## 5. Event analysis

sDARKEN darkens on serotonin binding → serotonin release events appear as **downward deflections** (negative % ΔF/F).  
Events are detected with a −2 SD threshold and a minimum inter-event spacing of 20 s.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import find_peaks
from scipy.stats import expon

# Re-apply style after seaborn (seaborn can reset rcParams)
apply_style()

FPS          = 0.5
SD_THRESHOLD = 2.0
MIN_DIST_F   = int(20 * FPS)

proc_recs = {}
events    = {}

for rec_id, rec in fusion.recordings.items():
    tr = detrend_linear(rec.traces.copy())
    tr = gaussian_smooth(tr, window_size=5, sigma=2.5)
    tr = to_percent(tr)
    proc_recs[rec_id] = Recording(
        name=f"Fusion {rec_id}",
        raw=tr, traces=tr, fps=rec.fps,
        excluded_rois=rec.excluded_rois,
        biological_replicate=rec.biological_replicate,
        condition=rec.condition,
    )
    events[rec_id] = {}
    for col in tr.columns:
        s = tr[col]
        min_h = -s.mean() + SD_THRESHOLD * s.std()
        peaks, _ = find_peaks(-s.values, height=min_h, distance=MIN_DIST_F)
        events[rec_id][col] = peaks

total = sum(len(v) for d in events.values() for v in d.values())
print(f"Detected {total} events  "
      f"(threshold: −{SD_THRESHOLD} SD, min spacing: {20} s)")
for rid in events:
    n = sum(len(v) for v in events[rid].values())
    print(f"  Fusion {rid}: {n} events across {len(events[rid])} ROIs")


In [ ]:
# ── Plot 5a: Event raster plot ───────────────────────────────────────────
colors    = plt.cm.Set2.colors
rec_ids   = list(events.keys())
cmap_     = {rid: colors[i % len(colors)] for i, rid in enumerate(rec_ids)}

fig, ax = plt.subplots(figsize=(12, 5))

y_pos   = 0
yticks  = []
ylabels = []
rec_y_centers = {}

for rid in rec_ids:
    y_start = y_pos
    for col, frames in events[rid].items():
        ax.scatter(frames / FPS, [y_pos] * len(frames),
                   marker='|', s=120, linewidth=1.8, color=cmap_[rid])
        yticks.append(y_pos)
        ylabels.append(col)
        y_pos += 1
    rec_y_centers[rid] = (y_start + y_pos - 1) / 2
    ax.axhspan(y_start - 0.5, y_pos - 0.5,
               color=cmap_[rid], alpha=0.08, zorder=0)
    y_pos += 1

ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(list(rec_y_centers.values()))
ax2.set_yticklabels([f"Fusion {rid}" for rid in rec_y_centers], fontsize=9)
ax2.spines['right'].set_visible(False)

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=7)
ax.set_xlabel("Time (s)")
ax.set_title("Spontaneous serotonin release events — raster plot")
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
save_fig(fig, "06_event_raster")
plt.show()


In [ ]:
# ── Plot 5b: Average event waveform (aligned to peak) ───────────────────
WINDOW_S = 30
WINDOW_F = int(WINDOW_S * FPS)

all_waves = []
per_rec   = {rid: [] for rid in events}

for rid in events:
    tr = proc_recs[rid].traces
    for col, frames in events[rid].items():
        s = tr[col].values
        for pf in frames:
            start, end = pf - WINDOW_F, pf + WINDOW_F + 1
            if start >= 0 and end <= len(s):
                all_waves.append(s[start:end])
                per_rec[rid].append(s[start:end])

if not all_waves:
    print("No complete waveforms found.")
else:
    t_wave = np.arange(-WINDOW_F, WINDOW_F + 1) / FPS
    mat    = np.array(all_waves)

    fig, ax = plt.subplots(figsize=(8, 5))
    for w in mat:
        ax.plot(t_wave, w, color='grey', alpha=0.12, lw=0.6)

    colors_ = plt.cm.Set2.colors
    for i, (rid, waves) in enumerate(per_rec.items()):
        if waves:
            m = np.array(waves).mean(axis=0)
            ax.plot(t_wave, m, color=colors_[i], lw=1.5, alpha=0.8,
                    label=f"Fusion {rid} (n={len(waves)})")

    grand_mean = mat.mean(axis=0)
    grand_sem  = mat.std(axis=0) / np.sqrt(len(mat))
    ax.plot(t_wave, grand_mean, color='black', lw=2.5,
            label=f"Grand mean  (n = {len(mat)} events)")
    ax.fill_between(t_wave, grand_mean - grand_sem, grand_mean + grand_sem,
                    color='black', alpha=0.25)

    ax.axhline(0, color='k', ls='--', lw=0.6)
    ax.axvline(0, color='red', ls='--', lw=1.0, label='Event peak')
    ax.set_xlabel("Time relative to event peak (s)")
    ax.set_ylabel("% ΔF/F")
    ax.set_title("Average sDARKEN event waveform")
    ax.legend(frameon=False, fontsize=8)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    save_fig(fig, "07_event_waveform")
    plt.show()


## 6. Replicate-level aggregation (CalciMetrics 2.0)

Average within each recording (biological replicate) first, then show N = 2 replicate means — the statistically correct representation when each iPSC fusion organoid is one independent observation.

In [ ]:
rep_means = fusion.combine_traces(level="replicate")
rep_means = detrend_linear(rep_means)
rep_means = gaussian_smooth(rep_means, window_size=5, sigma=2.5)
rep_means = to_percent(rep_means)

N = rep_means.shape[1]

rep_rec = Recording(
    name=f"replicate means (N = {N})",
    raw=rep_means,
    traces=rep_means,
    fps=fps,
    excluded_rois=[],
    biological_replicate="pooled",
    condition="sDARKEN_fusion",
)

fig = plot_mean_with_error(rep_rec, error="sd", ylabel="% ΔF/F")
save_fig(fig, "08_replicate_means")
plt.show()
